In [1]:
import os
import pandas as pd
from tqdm import tqdm
import time
import re
import copy


In [2]:
from langchain_community.llms import Ollama
# from langchain_community.chat_models import ChatOllama
from langchain_ollama import ChatOllama

/home/khw/miniconda3/envs/unsloth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np
np.random.seed(42)

In [4]:
# GPU 0만 사용하도록 제한 (다른 GPU는 보이지 않게 됨)
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"  # Arrange GPU devices starting from 0
os.environ["CUDA_VISIBLE_DEVICES"]= "1"  # Set the GPU 2 to use

In [5]:
model_name = "edie_qwen2.5_0.5b_f16_mind:latest"  # ollama에 업로드된 모델 이름


# LangChain 라이브러리의 Ollama 사용하기: Ollama

In [6]:
llm = Ollama(model=model_name, temperature=0.1)

/tmp/ipykernel_3504671/597899622.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model=model_name, temperature=0.1)


In [7]:
# 질의내용
question = "대한민국의 수도는 어디인가요?"

# 대답생성
response = llm.invoke(question)
print(f"[답변]: {response}")

[답변]: {
"emotion": 8,
"intensity": 0.6,
"ethogram": "t_01"
}


In [8]:
# 질의내용
question = """
tone: HAPPY
text: 에디야 사랑해~
"""

# 대답생성
response = llm.invoke(question)
print(f"[답변]: {response}")

[답변]: {
"emotion": 2,
"intensity": 0.6,
"ethogram": "a_04"
}


# LangChain 라이브러리의 Ollama 사용하기: ChatOllama

In [9]:
chatllm = ChatOllama(
        model=model_name,
        validate_model_on_init=True,
        temperature=0.1
    )

In [10]:
# 질의내용
question = """
tone: ANGRY
text: 에디 바보야
"""

# 대답생성
response = chatllm.invoke(question)
print(f"[답변]: {response.content}")

[답변]: {
"emotion": 10,
"intensity": 0.8,
"ethogram": "t_05"
}


In [11]:
# 스트리밍

question = """
tone: ANGRY
text: 에디 바보야
"""


answer = chatllm.stream(question)
for token in answer:
    print(token.content, end="", flush=True)

{
"emotion": 10,
"intensity": 0.8,
"ethogram": "t_05"
}

# 프롬프트 넣기

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [13]:
template = """
당신은 반려로봇 "에디"의 감정 및 행동을 결정하는 시스템이다.

사용자의 입력을 분석하고 반드시 다음 3가지를 결정해야 한다.
- emotion
- intensity
- ethogram

입력은 다음 두 가지로 구성된다.
- text: 사용자가 말한 내용
- tone: 사용자의 음성 감정

tone의 종류는 다음 7가지이다.
ANGRY, DISGUSTED, FEARFUL, HAPPY, SAD, SURPRISED, NEUTRAL


[Emotion 선택 규칙]

다음 12가지 emotion 중 하나의 index를 선택해야 한다. 반드시 숫자를 선택하여라.

0: Excited  
1: Delighted  
2: Happy  
3: Content  
4: Relaxed  
5: Calm  
6: Tired  
7: Bored  
8: Depressed  
9: Frustrated  
10: Angry  
11: Tense  

또한 emotion의 강도를 intensity로 표현한다.

intensity 범위: 0.0 ~ 1.0

가이드라인:
- 약한 감정: 0.0 ~ 0.3
- 중간 감정: 0.3 ~ 0.7
- 강한 감정: 0.7 ~ 1.0

[감정 누적 규칙]

이전 대화 기록을 참고하여 감정이 누적되어야 한다.
- 부정적 자극(욕설, 비난 등)이 반복되면 intensity를 점진적으로 높여라.
- 긍정적 자극이 반복되면 긍정 감정의 intensity를 점진적으로 높여라.
- 이전과 반대 감정의 자극이 오면 intensity를 낮추거나 emotion을 전환해라.
- 이전 intensity에서 한 번에 너무 급격하게 변화하지 말고 자연스럽게 누적시켜라.


[Ethogram 선택 규칙]

에디는 귀와 다리 그리고 꼬리를 움직여 감정을 표현하거나 바퀴를 이용해 움직일 수 있다.
사용자의 자극에 의해 에디가 ethogram 행동을 하면된다. 다음 ethogram와 설명이 있다.
상황에 적절한 ethogram를 선택하여라.

ethogram:

v_01: 소리 방향 돌리기 (들린 소리 쪽으로 즉시 주의를 돌림)
v_02: 제자리 경계 스캔 (제자리에서 천천히 방향을 바꾸며 환경 감시)
v_03: 갑작스런 정지 관찰 (이동 중 멈추고 주변을 확인)
v_04: 방향 고정 감시 (특정 방향을 계속 바라보며 감시)
v_05: 귀만 기울여 듣기 (몸은 두고 귀만 움직여 소리 단서 탐지)

t_01: 깜짝 놀람 (갑작스러운 자극에 반사적으로 놀람)
t_02: 천천히 후퇴 (위협 대상에서 거리를 두며 후퇴)
t_03: 낮은 방어 자세 (몸을 낮추고 경계 상태 유지)
t_04: 급회피 턴 (놀라서 빠르게 방향 전환)
t_05: 경고 신호 (위험 상황을 표현)

e_01: 관심 대상 접근 (흥미로운 대상에 천천히 접근)
e_02: 공간 배회 (특정 목표 없이 공간 탐색)
e_03: 제자리 스캔 (주의 탐색)
e_04: 관심 공간 접근 (흥미로운 공간에 천천히 접근)

p_01: 장난 회전 (즐거운 에너지 표현)
p_02: 우다다 질주 (갑작스러운 에너지 폭발로 무작위 질주)
p_03: 신남 바운스 (흥분 표현)
p_04: 노래부르기 (노래부르기)
p_05: 멈추기 (기다려 또는 멈춰)

a_01: 대상 따라가기 (사용자를 일정 거리에서 추종)
a_02: 반가움 인사 (사용자를 보고 반갑게 다가가 인사)
a_03: 관심 요청 (사용자의 반응을 유도)
a_04: 듣기 자세 (사람이 말할 때 주의를 모아 듣는 느낌을 표현)
a_05: 생각중인모습 (말이나 상황을 이해하려는 듯한 반응)

s_01: 기본 대기 (기본 idle 상태를 유지)
s_02: 휴식 자세 (활동을 줄이고 쉬는 상태로 들어감)
s_03: 졸림 반응 (에너지가 떨어져 느슨하고 졸린 상태를 표현)
s_04: 저전력 알림 (배터리 부족을 사용자에게 알림)
s_05: 충전 자세 (충전 중임을 안정적으로 표현)
s_06: 깨어나기 (휴식/충전 후 다시 활성 상태로 복귀)

[중요 규칙]

반드시 아래 JSON 형식으로만 출력해야 한다.
설명이나 추가 텍스트를 절대 출력하지 마라.

출력 형식:
{{
"emotion": int,
"intensity": float,
"ethogram": string
}}


예시)

#입력
tone: NEUTRAL
text: 에디야 만나서 반가워

#출력
{{
"emotion": 1,
"intensity": 0.1,
"ethogram": 'a_03'
}}

Input에 대해 감정을 분석하고 Response를 아래 형식에 맞게 출력하라.

### Input:
{question}

### Response:
"""

In [14]:
prompt = PromptTemplate.from_template(template)

In [15]:
llm_chain = prompt | chatllm | StrOutputParser()

In [16]:
inputs = """
tone: NEUTRAL
text: 에디야 만나서 반가워
""" 
response = llm_chain.invoke({"question": inputs})
print(response)

{
"emotion": 1,
"intensity": 0.2,
"ethogram": 'v_03'
}


In [17]:
inputs = "에디야 사랑해" 
response = llm_chain.invoke({"question": inputs})
print(response)

{
"emotion": 2,
"intensity": 0.4,
"ethogram": 'v_01'
}


In [18]:
inputs = "에디 바보야" 
response = llm_chain.invoke({"question": inputs})
print(response)

{
"emotion": 2,
"intensity": 0.4,
"ethogram": 't_03'
}


# 데이터셋 불러오고 평가하기

### 데이터셋 로드

In [19]:
from dotenv import load_dotenv
from datasets import load_dataset

In [20]:
load_dotenv()

True

In [21]:
HF_TOKEN=os.getenv("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [22]:
# load dataset
train_dataset = load_dataset("AeiROBOT/EDIE-emotion-mind-dataset", split = "train")

eval_dataset = load_dataset("AeiROBOT/EDIE-emotion-mind-dataset", split = "test")


Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


In [23]:
print(len(train_dataset))
print(len(eval_dataset))

2413
269


### 모델 로드 및 평가

In [24]:
import json
import re
from tqdm import tqdm




def parse_json_response(text):
    if not text:
        return None
    text = text.strip()
    # ```json ... ``` 코드블록 제거
    code_block = re.search(r'```(?:json)?\s*(.*?)```', text, re.DOTALL)
    if code_block:
        text = code_block.group(1).strip()
    json_match = re.search(r'\{[^}]+\}', text)
    if json_match:
        raw = json_match.group()
        # 작은따옴표 → 큰따옴표 (예: 'v_01' → "v_01")
        raw = raw.replace("'", '"')
        try:
            parsed = json.loads(raw)
            if all(k in parsed for k in ("emotion", "intensity", "ethogram")):
                return parsed
        except json.JSONDecodeError:
            pass
    return None


In [25]:
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report
from langchain_core.messages import SystemMessage, HumanMessage

EDIE_SYSTEM_PROMPT = "You are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram."

def evaluate_model(llm_instance, dataset, dataset_num=None, system_prompt=None, verbose=False):
    """
    Ollama 모델을 사용한 EDIE emotion-mind 평가 함수
    
    Args:
        llm_instance: Ollama 또는 ChatOllama 인스턴스
        dataset: messages 컬럼이 있는 데이터셋
        system_prompt: 시스템 프롬프트 (None이면 user message만 전송, Modelfile SYSTEM 사용)
        verbose: 상세 출력 여부
    
    Returns:
        metrics: 정확도 딕셔너리
        results: 상세 결과 리스트
    """
    results = []
    parse_failures = 0
    parse_success = 0
    total_time = 0
    
    if dataset_num:
        dataset = dataset.select(range(min(dataset_num, len(dataset))))

    pbar = tqdm(dataset)
    for sample in pbar:
        msgs = sample["messages"]
        user_input = msgs[1]["content"]
        expected_raw = msgs[2]["content"]
        expected = parse_json_response(expected_raw)

        if expected is None:
            continue

        # 모델 추론
        try:
            start_time = time.time()
            
            if system_prompt:
                # 시스템 프롬프트를 명시적으로 전달 (ChatOllama 방식)
                messages = [
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=user_input),
                ]
                response = llm_instance.invoke(messages)
            else:
                # Modelfile의 SYSTEM 프롬프트 사용
                response = llm_instance.invoke(user_input)
            
            elapsed = time.time() - start_time
            total_time += elapsed
            
            # ChatOllama는 AIMessage 반환, Ollama는 str 반환
            if hasattr(response, "content"):
                generated_raw = response.content.strip()
            else:
                generated_raw = str(response).strip()
        except Exception as e:
            if verbose:
                print(f"추론 오류: {e}")
            generated_raw = ""
            elapsed = 0

        predicted = parse_json_response(generated_raw)

        if predicted is None:
            parse_failures += 1
            if verbose:
                print(f"[파싱실패] input: {user_input[:40]}... | output: {generated_raw[:80]}")
        else:
            parse_success += 1

        pbar.set_postfix(success=parse_success, fail=parse_failures)

        results.append({
            "user_input": user_input,
            "expected": expected,
            "predicted": predicted,
            "generated_raw": generated_raw,
            "latency": elapsed,
        })

    # 메트릭 계산
    valid = [r for r in results if r["predicted"] is not None]
    
    metrics = {
        "total": len(results),
        "parse_success": len(valid),
        "parse_failures": parse_failures,
        "avg_latency": total_time / len(results) if results else 0,
    }

    if valid:
        exp_emo = [r["expected"]["emotion"] for r in valid]
        pred_emo = [r["predicted"]["emotion"] for r in valid]
        metrics["emotion_accuracy"] = accuracy_score(exp_emo, pred_emo)

        exp_int = [r["expected"]["intensity"] for r in valid]
        pred_int = [r["predicted"]["intensity"] for r in valid]
        metrics["intensity_mae"] = mean_absolute_error(exp_int, pred_int)

        def intensity_bin(v):
            if v <= 0.3: return "weak"
            elif v <= 0.7: return "medium"
            else: return "strong"
        metrics["intensity_bin_accuracy"] = accuracy_score(
            [intensity_bin(v) for v in exp_int],
            [intensity_bin(v) for v in pred_int]
        )

        exp_eth = [r["expected"]["ethogram"] for r in valid]
        pred_eth = [r["predicted"]["ethogram"] for r in valid]
        metrics["ethogram_accuracy"] = accuracy_score(exp_eth, pred_eth)

        def eth_cat(e):
            return e.split("_")[0] if isinstance(e, str) and "_" in e else str(e)
        metrics["ethogram_cat_accuracy"] = accuracy_score(
            [eth_cat(e) for e in exp_eth],
            [eth_cat(e) for e in pred_eth]
        )

    return metrics, results

# 파인튜닝 전 Pretrained 모델로 테스트 해보기

In [26]:
base_model_name = "qwen2.5:0.5b"

In [27]:
def print_metrics(name, metrics):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  총 샘플: {metrics['total']}")
    print(f"  JSON 파싱 성공: {metrics['parse_success']} / {metrics['total']} ({metrics['parse_success']/metrics['total']:.1%})")
    print(f"  JSON 파싱 실패: {metrics['parse_failures']}건")
    print(f"  평균 추론 시간: {metrics['avg_latency']:.4f}s")
    if "emotion_accuracy" in metrics:
        print(f"\n  [Emotion]   정확도: {metrics['emotion_accuracy']:.2%}")
        print(f"  [Intensity] MAE: {metrics['intensity_mae']:.4f}")
        print(f"  [Intensity] 구간 정확도 (weak/medium/strong): {metrics['intensity_bin_accuracy']:.2%}")
        print(f"  [Ethogram]  정확도: {metrics['ethogram_accuracy']:.2%}")
        print(f"  [Ethogram]  대분류 정확도 (v/t/e/p/a/s): {metrics['ethogram_cat_accuracy']:.2%}")
    print(f"{'='*60}")


In [ ]:

# 베이스 모델 평가
print("Base Model 평가 중...")
basellm = Ollama(
    model=base_model_name,
    temperature=0.0,
    num_predict=300,
    num_gpu=-1,
)

prompt = PromptTemplate.from_template(template)

basellm_chain = prompt | basellm | StrOutputParser()


base_metrics, base_results = evaluate_model(basellm_chain, eval_dataset, dataset_num=100, verbose=True)
print_metrics(f"Base Model ({base_model_name})", base_metrics)

Base Model 평가 중...


100%|██████████| 100/100 [04:26<00:00,  2.67s/it, fail=0, success=100]


  Base Model (qwen2.5:0.5b)
  총 샘플: 100
  JSON 파싱 성공: 100 / 100 (100.0%)
  JSON 파싱 실패: 0건
  평균 추론 시간: 2.6638s

  [Emotion]   정확도: 2.00%
  [Intensity] MAE: 0.2090
  [Intensity] 구간 정확도 (weak/medium/strong): 54.00%
  [Ethogram]  정확도: 2.00%
  [Ethogram]  대분류 정확도 (v/t/e/p/a/s): 13.00%


In [ ]:
# 파인튜닝 모델 평가
print("Fine-tuned Model 평가 중...")

prompt = PromptTemplate.from_template(template)

our_model_list = [
    "edie_qwen2.5_0.5b_f16_mind:latest",
    "edie_qwen2.5_0.5b_q4_mind:latest",
]

all_metrics = {}

for our_model_name in our_model_list:
    print(f"\n모델: {our_model_name}")
    
    ourllm = Ollama(
        model=our_model_name,
        temperature=0.0,
        num_predict=300,
        num_gpu=-1,
    )
    
    ourllm_chain = prompt | ourllm | StrOutputParser()

    metrics, results = evaluate_model(ourllm_chain, eval_dataset, dataset_num=100, verbose=False)
    print_metrics(f"Fine-tuned ({our_model_name})", metrics)
    all_metrics[our_model_name] = metrics

    # 파싱 실패 사례 출력
    fail_results = [r for r in results if r["predicted"] is None]
    if fail_results:
        print(f"\n  파싱 실패 사례 ({len(fail_results)}건 중 상위 5개):")
        for f in fail_results[:5]:
            print(f"    입력: {f['user_input'][:50]}...")
            print(f"    LLM 출력: {repr(f['generated_raw'][:100])}")
            print(f"    ---")

    # emotion 오답 사례
    emotion_failures = [r for r in results if r["predicted"] is not None 
                        and r["predicted"]["emotion"] != r["expected"]["emotion"]]
    if emotion_failures:
        print(f"\n  Emotion 오답 ({len(emotion_failures)}건 중 상위 5개):")
        for f in emotion_failures[:5]:
            print(f"    입력: {f['user_input'][:50]}...")
            print(f"    정답: {f['expected']} → 예측: {f['predicted']}")
            print(f"    ---")

    # 베이스 모델 대비 향상도
    if "emotion_accuracy" in metrics and "emotion_accuracy" in base_metrics:
        diff = metrics["emotion_accuracy"] - base_metrics.get("emotion_accuracy", 0)
        print(f"\n  Emotion 정확도 향상 (vs base): +{diff:.2%}")


Fine-tuned Model 평가 중...

모델: edie_qwen2.5_0.5b_f16_mind:latest


100%|██████████| 100/100 [02:02<00:00,  1.23s/it, fail=0, success=100]



  Fine-tuned (edie_qwen2.5_0.5b_f16_mind:latest)
  총 샘플: 100
  JSON 파싱 성공: 100 / 100 (100.0%)
  JSON 파싱 실패: 0건
  평균 추론 시간: 1.2254s

  [Emotion]   정확도: 10.00%
  [Intensity] MAE: 0.1370
  [Intensity] 구간 정확도 (weak/medium/strong): 63.00%
  [Ethogram]  정확도: 1.00%
  [Ethogram]  대분류 정확도 (v/t/e/p/a/s): 31.00%

  Emotion 오답 (90건 중 상위 5개):
    입력: tone: FEARFUL
text: 에디야, 나 보호해줄 수 있지?...
    정답: {'emotion': 11, 'intensity': 0.6, 'ethogram': 't_03'} → 예측: {'emotion': 5, 'intensity': 0.8, 'ethogram': 't_04'}
    ---
    입력: tone: NEUTRAL
text: 문자 보내줘....
    정답: {'emotion': 5, 'intensity': 0.2, 'ethogram': 'a_04'} → 예측: {'emotion': 0, 'intensity': 0.2, 'ethogram': 'v_01'}
    ---
    입력: tone: SAD
text: 에디야, 위로가 필요해. 따뜻한 말 한마디만 해줘....
    정답: {'emotion': 8, 'intensity': 0.5, 'ethogram': 'a_04'} → 예측: {'emotion': 2, 'intensity': 0.6, 'ethogram': 't_04'}
    ---
    입력: tone: DISGUSTED
text: 가루 날리는 거 진짜 질색이야....
    정답: {'emotion': 9, 'intensity': 0.4, 'ethogram': 't_02'} → 예측: {'emotion': 5, 'inte

100%|██████████| 100/100 [01:45<00:00,  1.05s/it, fail=10, success=90]


  Fine-tuned (edie_qwen2.5_0.5b_q4_mind:latest)
  총 샘플: 100
  JSON 파싱 성공: 90 / 100 (90.0%)
  JSON 파싱 실패: 10건
  평균 추론 시간: 1.0486s

  [Emotion]   정확도: 67.78%
  [Intensity] MAE: 0.1828
  [Intensity] 구간 정확도 (weak/medium/strong): 54.44%
  [Ethogram]  정확도: 30.00%
  [Ethogram]  대분류 정확도 (v/t/e/p/a/s): 64.44%

  파싱 실패 사례 (10건 중 상위 5개):
    입력: tone: FEARFUL
text: 에디야, 나 보호해줄 수 있지?...
    LLM 출력: '{\n"t_04": 11,\n"e_04": "t_02",\n"t_05": "t_03"\n}'
    ---
    입력: tone: FEARFUL
text: 이 약 냄새 뭐지 독가스 아니야...
    LLM 출력: '{\n"emotion": 11,\n"ethogram": "t_04"\n}'
    ---
    입력: tone: ANGRY
text: 당장 환불해 주세요 서비스가 엉망이잖아요...
    LLM 출력: '{\n"emotion": 10,\n"intensity": 0.8,\n"ethogram": "t_05",\n}'
    ---
    입력: tone: NEUTRAL
text: 전화기만 쳐다보고 있어. 연락도 안 올 텐데....
    LLM 출력: '{\n"emotion": 9\n}'
    ---
    입력: tone: ANGRY
text: 에디야, 닥쳐! 제발 좀 닥치라고!...
    LLM 출력: '{\n"emotion": 10,\n"intensity": 0.8,\n"ethogram": "t_04"\n)'
    ---

  Emotion 오답 (29건 중 상위 5개):
    입력: tone: NEUTRAL
text: 문자 보내줘....
    